# D470 - Slowly Changing Dimensions (SCD) Introduction

## Day 47: Understanding current state and historical state

A product changes category, a customer moves city, or an employee changes department. How should a data warehouse represent those changes?

In this lesson you will learn what SCD means, why it matters, and how **SCD Types 1, 2, 3, and 4** answer different business questions. Type 4 terminology varies, so both the separate-history-table convention and Kimball mini-dimensions are explained.

This is a conceptual notebook: read the examples in order. No database, Spark session, or package installation is required. The SQL snippets illustrate queries; they are not executable notebook cells.

## 1. Start with facts and dimensions

A **fact table** records business events and measurements, such as an order line, quantity, and actual selling price.

A **dimension table** describes the entities involved in those events, such as a product's name, category, and color, or a customer's city.

| Table | Example columns | Purpose |
|---|---|---|
| `fact_order_line` | order_id, order_time, product_sk, quantity, selling_price | What happened and how much? |
| `dim_product` | product_sk, product_id, product_name, category, color | Which product, with which attributes? |

`product_id` is the **business key** from the source system. `product_sk` is a warehouse **surrogate key** that identifies a dimension row. With Type 2, each version gets its own surrogate key.

## 2. What is SCD?

**SCD stands for Slowly Changing Dimension.** It is a set of data modeling techniques for handling changes to descriptive attributes in dimension tables over time.

Examples include a customer moving from Chennai to Bengaluru, a product moving from Premium to Standard, or an employee transferring from Sales to Operations.

?Slowly? reflects that these attributes often change less frequently than business transactions. It does not impose a fixed update frequency or mean the pipeline must run slowly.

The central design question is: **Should this attribute show only the latest value, or should we preserve its previous values?**

## 3. Why do we need SCD?

Suppose product `P101` belongs to **Premium** in January and moves to **Standard** on February 1.

| Order | Order date | Product | Actual order amount | Category at order time |
|---|---|---|---:|---|
| O1001 | 2026-01-15 | P101 | 100 | Premium |
| O1002 | 2026-02-10 | P101 | 90 | Standard |

Two valid reporting questions now exist:

- **Current classification:** What are total sales for products that are in Standard today? Both orders contribute: **190**.
- **Historical classification:** What were sales by category when each order occurred? Premium contributes **100** and Standard contributes **90**.

If the warehouse only overwrites the category, it cannot reconstruct the historical classification from that dimension alone. SCD makes the choice explicit and keeps reports consistent with the business requirement.

## 4. SCD Type 1: overwrite the old value

**Type 1 keeps the latest state by updating the existing dimension row.** The previous value is not retained in that dimension.

Use case: the product name was entered incorrectly as `Wirless Mouse`. The business wants the corrected spelling everywhere, including reports on earlier orders.

**Before correction**

| product_sk | product_id | product_name | category |
|---:|---|---|---|
| 101 | P101 | Wirless Mouse | Premium |

**After correction**

| product_sk | product_id | product_name | category |
|---:|---|---|---|
| 101 | P101 | Wireless Mouse | Premium |

The row and surrogate key remain the same. The misspelled name is overwritten.

## 5. How Type 1 processes changes

For each incoming business key:

1. If the key is new, insert a dimension row.
2. If it already exists and a managed attribute changed, update that row.
3. If its attributes are unchanged, leave the row unchanged.

```sql
-- Illustrative correction to an existing dimension row.
UPDATE dim_product_scd1
SET product_name = 'Wireless Mouse'
WHERE product_id = 'P101';
```

**Suitable uses:** correcting spelling, maintaining a current product catalog, or showing a customer's latest contact details when historical values are not required.

**Tradeoff:** simple storage and querying, but earlier values cannot be recovered from this table. Facts joined to it receive the latest attributes, so historical report groupings can change after an update. A separate audit log or source history may still exist elsewhere.

## 6. SCD Type 2: keep a new row for each version

**Type 2 preserves history by creating a new dimension row when a tracked attribute changes.** The old row is retained and marked as expired.

Use case: product `P101` moves from Premium to Standard on February 1. Analysts need each order attributed to the category that applied when it happened.

**Before the change**

| product_sk | product_id | product_name | category | valid_from | valid_to | is_current |
|---:|---|---|---|---|---|---|
| 101 | P101 | Wireless Mouse | Premium | 2026-01-01 | NULL | true |

**After the change**

| product_sk | product_id | product_name | category | valid_from | valid_to | is_current |
|---:|---|---|---|---|---|---|
| 101 | P101 | Wireless Mouse | Premium | 2026-01-01 | 2026-02-01 | false |
| 102 | P101 | Wireless Mouse | Standard | 2026-02-01 | NULL | true |

There is one business entity (`P101`) and two historical versions (`101` and `102`). Only the second version is current.

## 7. Understanding Type 2 columns and time boundaries

| Column | Meaning |
|---|---|
| `product_sk` | Unique identifier for one dimension version |
| `product_id` | Stable business key connecting versions of the same product |
| `valid_from` | Time this version becomes effective, inclusive |
| `valid_to` | Time this version stops being effective, exclusive; NULL means open-ended |
| `is_current` | Whether this is the current version |

Use **half-open intervals**: `valid_from <= event_time < valid_to`. This avoids matching two versions at a change boundary.

```text
2026-01-01                 2026-02-01
     [ Premium, SK 101          )
                               [ Standard, SK 102 ...
```

An order exactly at `2026-02-01 00:00:00` uses Standard. Dates in these examples represent midnight in one agreed timezone. Real pipelines should define both timestamp precision and timezone.

The effective time should reflect the agreed business meaning. Source commit time and warehouse load time are not automatically the same as business effective time.

## 8. How Type 2 processes changes

For an incoming product change effective at time `T`:

1. Find the current row using the business key.
2. For a new product, insert its first version with a new surrogate key.
3. For an existing product, compare the **tracked attributes** with the current row.
4. If they changed, expire the current row: set `valid_to = T` and `is_current = false`.
5. Insert the new version with a new surrogate key, `valid_from = T`, `valid_to = NULL`, and `is_current = true`.
6. If the tracked attributes did not change, create no new version.

For the category change, step 4 expires row 101 and step 5 inserts row 102. Updating only an ingestion timestamp should not create a new business version.

Production implementations must handle the expire/insert pair consistently and make retries safe. Changes arriving late may require repairing earlier intervals; simply appending them as the latest version can produce incorrect history.

## 9. Connect facts to the correct version

When loading a fact, look up the dimension version whose validity interval contains the order time, then store that version's surrogate key.

| order_id | order_time | product_sk | actual_order_amount |
|---|---|---:|---:|
| O1001 | 2026-01-15 | 101 | 100 |
| O1002 | 2026-02-10 | 102 | 90 |

```sql
SELECT d.category, SUM(f.actual_order_amount) AS sales
FROM fact_order_line f
JOIN dim_product_scd2 d ON f.product_sk = d.product_sk
GROUP BY d.category;
```

**Historical result:** Premium = 100; Standard = 90.

Do not add `WHERE d.is_current = true` to this historical query: that would remove facts linked to expired versions. Joining only on `product_id` without selecting a version can duplicate facts across all historical rows.

Actual selling prices and amounts belong in the fact table. A product's historical list price can provide context, but must not replace the price actually charged.

## 10. Look up a version using an event timestamp

If staged orders contain the business key instead of a dimension surrogate key, an interval lookup identifies the historical version:

```sql
SELECT o.order_id, d.product_sk, d.category
FROM staged_orders o
JOIN dim_product_scd2 d
  ON o.product_id = d.product_id
 AND o.order_time >= d.valid_from
 AND (o.order_time < d.valid_to OR d.valid_to IS NULL);
```

For a current catalog, the query is simpler:

```sql
SELECT product_id, product_name, category
FROM dim_product_scd2
WHERE is_current = true;
```

Each order should match exactly one applicable version. Overlapping intervals create duplicate matches; missing dimension history creates unmatched orders. A production fact loader needs an explicit policy for unmatched rows, such as an unknown member or deferred lookup, rather than silently dropping them.

## 11. Type 1 and Type 2 compared

| Question | SCD Type 1 | SCD Type 2 |
|---|---|---|
| What happens on a change? | Overwrite the existing row | Expire the old version and insert a new row |
| What does the table retain? | Latest attributes | Historical and current versions |
| Rows per business key | Usually one | One per tracked version |
| Surrogate key on update | Keep existing key | Assign a new key to the new version |
| Validity columns needed? | Not for basic Type 1 | Usually start, end, and current indicator |
| Historical category reporting | Uses current category | Can use category effective at event time |
| Storage and loading | Smaller and simpler | More rows and more loading logic |
| Example | Correct a product-name typo | Preserve product-category or customer-region changes |

Choose based on the business question. Type 2 is useful when past attributes matter; Type 1 is appropriate when the latest or corrected value is sufficient.

A dimension may apply different policies to different attributes. For example, category may use Type 2 while spelling corrections use Type 1. Define whether a correction should update all historical versions of that attribute.

## 12. SCD Type 3: keep selected previous values in columns

**Type 3 adds columns to retain a limited number of earlier or alternate attribute values in the same dimension row.** A common design stores the current and immediately previous category.

Use case: a merchandising team wants to compare sales using the current category and the previous category after a reclassification.

| State | product_sk | product_id | previous_category | current_category | category_changed_at |
|---|---:|---|---|---|---|
| Initial | 101 | P101 | NULL | Premium | NULL |
| After February 1 change | 101 | P101 | Premium | Standard | 2026-02-01 |
| After March 1 change | 101 | P101 | Standard | Budget | 2026-03-01 |

These are snapshots of **one row over time**, not three retained rows. On each genuine change, copy the old current value into `previous_category`, assign the new current value, and update the change date. Ignore unchanged inputs so they do not erase the previous value.

After the March change, Premium is no longer available in this two-value design. The surrogate key remains 101 throughout.

**Benefit:** easy comparisons using selected classifications, with no extra dimension versions. **Limitation:** fixed history depth; this cannot answer arbitrary historical as-of questions. Some designs preserve an original or fixed alternate classification instead of shifting the immediately previous value. Define the column meaning explicitly.

Reference: [Kimball's description of Types 1, 2, and 3](https://www.kimballgroup.com/2008/08/slowly-changing-dimensions/).

## 13. SCD Type 4: current table plus a separate history table

**Terminology note:** ?Type 4? is also used for the separate-history-table pattern shown here. Kimball uses Type 4 to mean a **mini-dimension**, explained next. State the pattern when discussing Type 4 so the table design is unambiguous.

In the separate-history-table pattern, one table contains current values while another retains prior versions. Use case: a product catalog needs a compact current-state table, and analysts need historical product classifications.

After P101 changes from Premium to Standard on February 1:

**`dim_product_current`**

| product_sk | product_id | category | valid_from |
|---:|---|---|---|
| 101 | P101 | Standard | 2026-02-01 |

**`dim_product_history`**

| history_sk | product_id | category | valid_from | valid_to |
|---:|---|---|---|---|
| 9001 | P101 | Premium | 2026-01-01 | 2026-02-01 |

For each genuine tracked change:

1. Copy the outgoing current state into history, preserving its start time and closing it at the change time.
2. Update the current row with the incoming values and new start time.
3. Make both writes consistent and retries safe to avoid missing or duplicate history.

This example stores **expired versions only** in history. A historical lookup must consider both tables, combining their attributes and validity intervals with `UNION ALL`, then applying the event-time interval condition. Other designs store every version in history, including the current one; document the chosen convention.

**Tradeoff:** current-state queries stay simple, but historical queries and coordinated writes are more complex. Joining an old fact only to the current row still returns today's attributes. Unlike Type 2's single versioned dimension, history is physically separated, so define the historical lookup and fact-key strategy explicitly.

## 14. Kimball Type 4: split rapidly changing attributes into a mini-dimension

Kimball Type 4 moves a group of frequently changing attributes into a small **mini-dimension**. This avoids repeatedly copying a wide base dimension for profile changes.

Use case: customer identity changes rarely, while engagement and spending bands change often.

**Base dimension**

| customer_sk | customer_id | customer_name |
|---:|---|---|
| 201 | C01 | Anu |

**Profile mini-dimension: reusable combinations**

| profile_sk | engagement_band | spending_band |
|---:|---|---|
| 501 | Medium | Low |
| 502 | High | Medium |

**Facts retain the profile observed at event time**

| order_id | customer_sk | profile_sk | amount |
|---|---:|---:|---:|
| O2001 | 201 | 501 | 100 |
| O2002 | 201 | 502 | 150 |

When Anu's profile changes, later facts use profile 502; earlier facts retain 501. Reuse an existing profile combination or insert a new one. Do not overwrite a combination referenced by older facts.

Different customers can share a profile row. This is not an archive of every customer version. Facts record profiles at event times; tracking changes between events requires additional history. Use bands rather than unconstrained values to limit combination growth.

Reference: [Kimball Type 4: Add Mini-Dimension](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/type-4-mini-dimension/).

## 15. Choosing among SCD Types 1, 2, 3, and 4

| Strategy | Where are changed values stored? | History available | Example use case |
|---|---|---|---|
| Type 1 | Overwrite the same row | Latest state only | Correct a product-name typo |
| Type 2 | Add version rows in the same dimension | Captured, retained tracked versions | Category at the time of an order |
| Type 3 | Add selected previous/alternate columns | Limited to the columns retained | Compare current and previous classifications |
| Type 4: separate history convention | Current table plus history table | Captured versions across tables | Current catalog with a historical archive |
| Type 4: Kimball mini-dimension | Separate attribute-profile combinations | Profiles referenced by facts at event time | Frequently changing customer segments |

Decide which business questions must be supported, how much history is needed, and how facts will find the correct attributes. The SCD number alone is insufficient when Type 4 naming differs.

## 16. SCD and CDC solve different problems

**Change Data Capture (CDC)** identifies source changes, such as inserts, updates, and deletes. **SCD** determines how a dimension represents those changes.

```text
Source product update
        |
        v
CDC event with before/after values
        |
        +--> Type 1 dimension: update current attributes
        |
        +--> Type 2 dimension: expire old version, insert new version
```

CDC can feed either strategy. SCD can also be implemented by comparing periodic snapshots, but snapshots cannot recover intermediate changes that were never observed.

For Type 2, preserve meaningful changes in order instead of collapsing a batch to only its latest row. Define how deletes behave: one policy is to close the current version without opening another. Under that policy, active products have one current row and deleted products have none.

## 17. Check your understanding

1. A product name contains a typo. The business wants the correction applied everywhere. Which type fits?
2. A customer moves from Chennai to Bengaluru. Reports must show revenue by the customer's city at purchase time. Which type fits?
3. A product has three genuine tracked states over time. How many rows does each strategy retain?
4. A repeated event contains exactly the same tracked values. Should Type 2 create a new version?
5. Why can joining a Type 2 dimension only on the business key inflate revenue?

6. A Type 3 row changes from Premium to Standard to Budget, retaining only current and previous category. Which value is lost?
7. In the separate-history-table pattern, why is the current table alone insufficient for an as-of report?
8. In Kimball Type 4, can two customers share the same profile row?

### Suggested answers

1. **Type 1:** overwrite the incorrect name.
2. **Type 2:** preserve city versions and associate purchases with the applicable version.
3. **Type 1: one row. Type 2: three rows**, assuming all changes were captured and history is retained.
4. **No.** Unchanged tracked values do not represent a new business state; retries should also be idempotent.
5. A fact can match multiple versions of the same business key, repeating its amount in the aggregation.
6. **Premium.** The remaining values are Budget (current) and Standard (previous).
7. Expired states are in the history table; a historical lookup must include the relevant retained versions.
8. **Yes.** Profile rows describe reusable attribute combinations; facts associate each customer with a profile at event time.

## 18. Continue with the Day 47 labs

The examples here are illustrative. Types 3 and 4 are conceptual notes; the existing implementation labs focus on CDC, Type 1, and Type 2. The following notebooks use their own product IDs and datasets to implement these concepts:

1. [D471 - MySQL CDC Source](D471_MySQL_CDC_Source.ipynb): create product changes and capture their source events.
2. [D472 - Spark / Iceberg SCD1](D472_Spark_Iceberg_SCD1.ipynb): maintain the latest product attributes.
3. [D473 - Spark / Iceberg SCD2](D473_Spark_Iceberg_SCD2.ipynb): preserve product versions and validate history.

As you work through the labs, observe the business key, version key, tracked attributes, validity boundaries, and behavior when the same input is processed again.